# Consumer Feedback & Unseen-Template Evaluation

**Pregunta:** ¿qué tan bien generaliza un clasificador entrenado con textos generados?

**SIMULACIÓN independiente:** 650 reseñas bilingües creadas con plantillas, semilla 2240. Las etiquetas no provienen de anotación humana. No se unen estas reseñas a facturas UCI ni se inventa una relación entre sentimiento y ventas.

## 1. Preparación

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from a checkout of retail-analytics-pipeline.")
sys.path.insert(0, str(ROOT / "src"))
REPORTS = ROOT / "reports"


**Dónde:** Jupyter en Codespaces o una copia del repositorio. La carpeta raíz se identifica por `pyproject.toml`; los mismos módulos se usan desde `main.py`.

## 2. Crear y revisar feedback

In [2]:
from market_insights.data import simulate_feedback
feedback = simulate_feedback()
display(feedback.sentiment_label.value_counts())
display(feedback[['review_text','sentiment_label','template_id']].head())

sentiment_label
positive    273
negative    215
neutral     162
Name: count, dtype: int64

,review_text,sentiment_label,template_id
0,La calidad es buena y el precio razonable. Lo ...,positive,La calidad es buena y el precio razonable.
1,El empaque falló y el contenido decepciona. I ...,negative,El empaque falló y el contenido decepciona.
2,No me gustó y no lo recomiendo. Lo probé duran...,negative,No me gustó y no lo recomiendo.
3,"Me encantó, cumplió mis expectativas. Lo usé d...",positive,"Me encantó, cumplió mis expectativas."
4,"It's ok, nothing special. Lo encontré en el su...",neutral,"It's ok, nothing special."


**Stack:** NumPy y pandas. Se conservan 273 positive, 215 negative y 162 neutral. Cada texto completo es único, pero comparte frases base con otras reseñas. `template_id` permite detectar esa dependencia.

## 3. Limpieza y evaluación

In [3]:
from market_insights.sentiment import clean_text, evaluate_sentiment
print(clean_text('No, no me gustó!'))
metrics, heldout, matrix, model = evaluate_sentiment(feedback)
display(metrics.round(3))

no no me gustó


,split,model,train_n,test_n,accuracy,macro_f1,f1_positive,f1_negative,f1_neutral
0,random_stratified,majority_baseline,520,130,0.423,0.198,0.595,0.000,0.000
1,random_stratified,logreg_unigrams,520,130,0.908,0.907,0.909,0.916,0.896
2,random_stratified,logreg_bigrams,520,130,0.923,0.922,0.930,0.914,0.923
3,random_stratified,naive_bayes,520,130,0.931,0.930,0.939,0.914,0.938
4,random_stratified,logreg_word_char,520,130,0.892,0.891,0.897,0.914,0.861
5,unseen_templates,majority_baseline,503,147,0.122,0.073,0.000,0.218,0.000
6,unseen_templates,logreg_unigrams,503,147,0.218,0.143,0.364,0.026,0.038
7,unseen_templates,logreg_bigrams,503,147,0.197,0.135,0.344,0.024,0.038
8,unseen_templates,naive_bayes,503,147,0.238,0.158,0.410,0.025,0.038
9,unseen_templates,logreg_word_char,503,147,0.293,0.278,0.375,0.095,0.364


**Stack:** TF-IDF de palabras/caracteres, Logistic Regression, MultinomialNB y baseline mayoritario. Se conservan acentos y negación. El vocabulario se aprende solo con train. Se comparan un split estratificado y un split por familias de plantillas: una frase base de test no aparece en train.

## 4. Matriz de confusión y errores

In [4]:
from market_insights.visualization import plot_sentiment
plot_sentiment(metrics, matrix, REPORTS/'figures')
display(heldout.loc[heldout.sentiment_label.ne(heldout.predicted)].head(8))

,feedback_id,review_text,sentiment_label,template_id,predicted
13,FB-0014,Very happy with this purchase. It was part of ...,positive,Very happy with this purchase.,negative
23,FB-0024,Good packaging but average quality. I tried it...,neutral,Good packaging but average quality.,negative
26,FB-0027,Ok. Fue una compra para el trabajo. Nuevo form...,positive,Ok.,neutral
31,FB-0032,Poor quality and disappointing service. I boug...,negative,Poor quality and disappointing service.,positive
36,FB-0037,Very happy with this purchase. Lo probé durant...,positive,Very happy with this purchase.,negative
41,FB-0042,Very happy with this purchase. It was part of ...,positive,Very happy with this purchase.,negative
45,FB-0046,Poor quality and disappointing service. Lo ele...,negative,Poor quality and disappointing service.,positive
48,FB-0049,Good packaging but average quality. It was a w...,neutral,Good packaging but average quality.,positive


**Stack:** pandas y Matplotlib. La matriz corresponde al candidato fijado Logistic Regression con palabras y caracteres sobre familias no vistas. El modelo de entrega está fijado, no elegido por el mejor test. Revisar F1 por clase y ejemplos; no considerar la accuracy aleatoria como prueba de rendimiento en reseñas reales.

![Figura de la ejecución registrada](../reports/figures/sentiment_confusion.png)

## 5. Model card y próximo experimento

In [5]:
display(metrics.loc[metrics.model.eq('logreg_word_char'), ['split','accuracy','macro_f1','f1_positive','f1_negative','f1_neutral']])

,split,accuracy,macro_f1,f1_positive,f1_negative,f1_neutral
4,random_stratified,0.892308,0.890629,0.897196,0.913580,0.861111
9,unseen_templates,0.292517,0.277958,0.375000,0.095238,0.363636


El buen resultado del split aleatorio se reduce mucho con familias no vistas. Próximo paso: etiquetas humanas y texto real de una fuente con permiso de uso, además de validación por idioma y dominio. No se presenta esta simulación como un clasificador listo para producción. La mejora del diagnóstico de familias no vistas llega a 29.3%, todavía insuficiente. El generador asigna etiquetas antes de elegir textos ambiguos; no es un conjunto anotado por humanos.